In [1]:
from langgraph.graph import StateGraph ,START , END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(
    model_name="llama-3.3-70b-versatile"
)

In [2]:
class BlogState(TypedDict):
    title: str
    outline:str
    content: str
    

In [3]:
def create_outline(state:BlogState) -> BlogState:
    title = state["title"]

    #calling the gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'

    outline = model.invoke(prompt).content

    #update the state
    state["outline"] =  outline

    return state


In [4]:
def create_blog(state: BlogState) -> BlogState:
    title = state["title"]
    outline = state["outline"]

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content
    state["content"] = content

    return state

In [5]:
graph = StateGraph(BlogState)


graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START , 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()


In [ ]:
initial_state = {"title":"Rise of Ai in India"}

final_state = workflow.invoke(initial_state)

print(final_state)


print("===========")

print(final_state["content"])

{'title': 'Ris of Ai in India', 'outline': 'Here is a detailed outline for a blog on the topic "Rise of AI in India":\n\n**Title:** "The Rise of AI in India: Opportunities, Challenges, and Future Prospects"\n\n**Introduction**\n\n* Brief overview of Artificial Intelligence (AI) and its growing importance globally\n* Introduction to the Indian context: India\'s growing economy, tech-savvy population, and increasing adoption of AI\n* Thesis statement: India is poised to become a significant player in the global AI landscape, with both opportunities and challenges arising from the rise of AI in the country.\n\n**Section 1: Opportunities for AI in India**\n\n* **Economic Growth**: How AI can contribute to India\'s economic growth, including:\n\t+ Increased productivity and efficiency in industries such as manufacturing, healthcare, and finance\n\t+ Creation of new job opportunities in AI-related fields\n\t+ Potential for AI to drive innovation and entrepreneurship in India\n* **Social Impa